## Téléchargement du dataset

In [1]:
# Import des librairies nécessaires

from tqdm import tqdm
import time

print("🚀 Préparation du téléchargement des librairies...")

for _ in tqdm(range(30), desc="Import en cours"):
    time.sleep(0.05)  # simulation de chargement

from ucimlrepo import fetch_ucirepo 
import pandas as pd
from pathlib import Path

print("✅ Librairies importées avec succès !")

# Définir le chemin du dossier et du fichier
data_folder = Path.cwd().parent / 'data'

print("📥 Téléchargement des datasets en cours...")

# Animation de progression simulée pendant le téléchargement
for _ in tqdm(range(50), desc="Téléchargement en cours"):
    time.sleep(0.04)  # simulation d’attente

# Récupération directe depuis UCI
dataset = fetch_ucirepo(id=697) 
X = dataset.data.features
y = dataset.data.targets
df = pd.concat([X, y], axis=1)

print("🎉 Téléchargement des datasets terminé !")

file_path = data_folder / "data_brut.csv"

# Sauvegarder dans un fichier CSV local
df.to_csv(file_path, sep=';', index=False, encoding='utf-8')

print(f"💾 Fichier bien enregistré dans : {file_path}")

🚀 Préparation du téléchargement des librairies...


Import en cours: 100%|██████████| 30/30 [00:01<00:00, 19.47it/s]


✅ Librairies importées avec succès !
📥 Téléchargement des datasets en cours...


Téléchargement en cours: 100%|██████████| 50/50 [00:02<00:00, 24.33it/s]


🎉 Téléchargement des datasets terminé !
💾 Fichier bien enregistré dans : c:\Users\romua\Documents\La_Plateforme_\Projet 5 - Perceptron Multicouche\ANN-playground\data\data_brut.csv


In [3]:
# Définir le chemin du dossier et du fichier
file_path = data_folder / "data_brut.csv"

df = pd.read_csv(file_path, sep=';')

# Affichage des informations générales sur le DataFrame
print("Infos sur le DataFrame :")
df.info()

# Vérification des valeurs manquantes dans chaque colonne
print("\nValeurs manquantes par colonne :")
print(df.isnull().sum())

# Verification des doublons
print("\nNombre de doublons dans le DataFrame :")
print(df.duplicated().sum())


# Suppression des lignes où "Mother's occupation" est 125, 173 ou 191
codes_to_remove = [125, 173, 191]

# Nombre de lignes avant suppression
n_before = len(df)

# Suppression des lignes
df_clean = df[~df["Mother's occupation"].isin(codes_to_remove)].copy()

# Nombre de lignes après suppression
n_after = len(df_clean)

# Nombre de lignes supprimées
n_removed = n_before - n_after

print(f"Nombre de lignes supprimées : {n_removed}")
print(f"Pourcentage de lignes supprimées : {n_removed / n_before * 100:.2f}%")

df = df_clean

Infos sur le DataFrame :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital Status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualifi

# Ajout de features

In [ ]:
### Dictionnaire des groupes de qualifications ###

qualification_groups = {
    0: {
        4 : 'Diplôme supérieur de troisième cycle (Doctorat)',
        43 : 'Enseignement supérieur - doctorat (3ᵉ cycle)',
        3 : 'Diplôme supérieur de deuxième cycle (Master)',
        2 : 'Diplôme supérieur de premier cycle (Licence/Bachelor)',
        40 : 'Diplôme de premier cycle universitaire',
        6 : 'Autres formations supérieures'
    },
    1: {
        42 : 'Diplôme technique professionnel supérieur',
        39 : 'Diplôme de spécialisation technologique',
        5 : 'Formation professionnelle continue'
    },
    2: {
        1 : 'Enseignement secondaire',
        9 : "12ᵉ année d'études non complétée",
        10 : "11ᵉ année d'études non complétée",
        12 : 'Autres formations de 11ᵉ année',
        14 : '10ᵉ année',
        15 : '10ᵉ année non complétée'
    },
    3: {
        19 : 'Éducation de base 3ᵉ cycle (9ᵉ/10ᵉ/11ᵉ année)',
        38 : 'Éducation de base 2ᵉ cycle (6ᵉ/7ᵉ/8ᵉ année)'
    }
}

def map_qualification_group(code):
    try:
        code = int(code)
    except:
        return None
    for group, codes in qualification_groups.items():
        if code in codes:
            return group
    return None

df['Previous qualification Group'] = df['Previous qualification'].apply(map_qualification_group)

# Réorganiser les colonnes pour insérer la nouvelle juste après "Previous qualification"
cols = list(df.columns)
idx = cols.index('Previous qualification')
cols.insert(idx+1, cols.pop(cols.index('Previous qualification Group')))
df = df[cols]


### Dictionnaire de classification des groupes de nationalités ###

nationality_groups = {
    0: {
        2: 'Elite Européenne Occidentale',
        13: 'Elite Européenne Occidentale',
        14: 'Elite Européenne Occidentale',
        1: 'Europe du Sud Développée',
        6: 'Europe du Sud Développée',
        11: 'Europe du Sud Développée',
    },
    1: {
        17: 'Europe de l Est Intégrée',
        62: 'Europe de l Est Intégrée',
        32: 'Puissances Émergentes',
        105: 'Puissances Émergentes',
    },
    2: {
        41: 'Amérique Latine Émergente',
        101: 'Amérique Latine Émergente',
        109: 'Amérique Latine Émergente',
        100: 'Europe de l Est en Transition',
        103: 'Europe de l Est en Transition',
        108: 'Systèmes Autoritaires Spécifiques',
    },
    4: {
        22: 'Îles Atlantiques Stabilisées',
        26: 'Îles Atlantiques Stabilisées',
        21: 'Géants Africains Ressources Naturelles',
        25: 'Géants Africains Ressources Naturelles',
        24: 'Petits États Fragiles',
    }
}

# Fonction pour mapper le code nationalité vers le groupe
def map_nationality_group(code):
    try:
        code = int(code)
    except:
        return None
    for group, codes in nationality_groups.items():
        if code in codes:
            return group
    return None

# Appliquer la fonction sur la colonne 'Nacionality'
df['Nationality Group'] = df['Nacionality'].apply(map_nationality_group)

# Insérer la nouvelle colonne juste après 'Nacionality'
cols = list(df.columns)
idx = cols.index('Nacionality')
cols.insert(idx + 1, cols.pop(cols.index('Nationality Group')))
df = df[cols]

### Dictionnaire de classification des groupes de qualification parents ###

parents_education_groups = {
    0: {
        5: "Doctorat (3ᵉ cycle)",
        44: "Enseignement supérieur - doctorat (3ᵉ cycle)",
        4: "Master (2ᵉ cycle)",
        43: "Enseignement supérieur - master (2ᵉ cycle)",
        3: "Diplôme supérieur deuxième cycle",
        41: "Cours d'études supérieures spécialisées",
        40: "Enseignement supérieur - diplôme (1er cycle)",
        2: "Diplôme supérieur premier cycle"
    },
    1: {
        1: "Enseignement secondaire (12ᵉ année ou équivalent)",
        9: "12ᵉ année non complétée",
        10: "11ᵉ année non complétée",
        12: "Autres formations 11ᵉ année",
        13: "2ᵉ année complémentaire lycée",
        14: "10ᵉ année",
        20: "Formation complémentaire lycée",
        22: "Formation technique professionnelle",
        27: "2ᵉ cycle du lycée général",
        39: "Cursus de spécialisation technologique",
        42: "Cours technique supérieur professionnel"
    },
    2: {
        19: "Éducation de base 3ᵉ cycle",
        25: "Formation complémentaire non complétée",
        26: "7ᵉ année",
        29: "9ᵉ année non complétée",
        30: "8ᵉ année",
        31: "Cours généraux administration et commerce",
        33: "Comptabilité et administration",
        37: "Éducation de base 1er cycle",
        38: "Éducation de base 2ᵉ cycle",
        18: "Commerce général",
        6: "Formation continue"
    },
    3: {
        11: "7ᵉ année (ancienne échelle)",
        34: "Inconnu",
        35: "Analphabète",
        36: "Lecture sans formation complète"
    }
}

# Fonction pour mapper le code nationalité vers le groupe
def map_parents_education_group(code):
    try:
        code = int(code)
    except:
        return None
    for group, codes in parents_education_groups.items():
        if code in codes:
            return group
    return None

# Appliquer la fonction sur la colonne 'Mothers qualification' et 'Fathers qualification'
df["Mother's qualification Group"] = df["Mother's qualification"].apply(map_parents_education_group)
df["Father's qualification Group"] = df["Father's qualification"].apply(map_parents_education_group)

# Insérer la nouvelle colonne juste après 'Mother's qualification'
cols = list(df.columns)
idx = cols.index("Mother's qualification")
cols.insert(idx + 1, cols.pop(cols.index("Mother's qualification Group")))
df = df[cols]

# Insérer la nouvelle colonne juste après 'Father's qualification'
cols = list(df.columns)
idx = cols.index("Father's qualification")
cols.insert(idx + 1, cols.pop(cols.index("Father's qualification Group")))
df = df[cols]

### Dictionnaire de classification des groupes de profession des parents ###

parents_occupation_groups = {
    0: {
        1: "Représentants du pouvoir législatif, directeurs et cadres dirigeants",
        2: "Spécialistes des activités intellectuelles et scientifiques",
        101: "Officiers des forces armées",
        121: "Spécialistes des sciences physiques, mathématiques, ingénierie",
        122: "Professionnels de la santé",
        123: "Enseignants",
        124: "Spécialistes en finance, comptabilité, organisation, relations publiques"
    },
    1: {
        3: "Techniciens et professions de niveau intermédiaire",
        102: "Sous-officiers des forces armées",
        112: "Directeurs des services administratifs et commerciaux",
        114: "Directeurs de l'hôtellerie, restauration, commerce et services",
        131: "Techniciens et professions intermédiaires des sciences et ingénierie",
        132: "Techniciens et professionnels de santé de niveau intermédiaire",
        134: "Techniciens intermédiaires services juridiques, sociaux, sportifs, culturels",
        135: "Techniciens en technologies de l'information et de la communication"
    },
    2: {
        4: "Personnel administratif",
        103: "Autre personnel des forces armées",
        141: "Employés de bureau, secrétaires, opérateurs de saisie",
        143: "Opérateurs de services de données, comptabilité, statistiques",
        144: "Personnel de soutien administratif",
        154: "Personnel des services de protection et de sécurité",
        174: "Ouvriers qualifiés en électricité et électronique"
    },
    3: {
        5: "Travailleurs des services personnels, sécurité et ventes",
        151: "Travailleurs des services personnels",
        152: "Vendeurs",
        153: "Travailleurs des soins personnels et assimilés",
        161: "Agriculteurs et ouvriers qualifiés de la production agricole et animale",
        171: "Ouvriers qualifiés du bâtiment et assimilés (sauf électriciens)",
        172: "Ouvriers qualifiés métallurgie, travail des métaux",
        175: "Travailleurs transformation alimentaire, bois, habillement, artisanats",
        181: "Opérateurs d'installations fixes et machines",
        182: "Travailleurs de l'assemblage",
        183: "Conducteurs de véhicules et opérateurs d'équipements mobiles"
    },
    4: {
        6: "Agriculteurs et ouvriers qualifiés agriculture, pêche et sylviculture",
        7: "Ouvriers qualifiés industrie, construction et artisans",
        8: "Opérateurs d'installation, machines et travailleurs de l'assemblage",
        163: "Agriculteurs, éleveurs, pêcheurs, chasseurs de subsistance",
        192: "Travailleurs non qualifiés agriculture, production animale, pêche",
        193: "Travailleurs non qualifiés industrie extractive, construction, fabrication",
        194: "Aides à la préparation des repas",
        195: "Vendeurs ambulants (hors produits alimentaires) et prestataires de rue"
    },
    5: {
        9: "Travailleurs non qualifiés"
    },
    6: {
        0: "Étudiant (en formation)",
        10: "Professions des forces armées (hiérarchie variable)",
        90: "Autre situation",
        99: "(vide)"
    }
}

# Fonction pour mapper le code nationalité vers le groupe
def map_parents_occupation_group(code):
    try:
        code = int(code)
    except:
        return None
    for group, codes in parents_occupation_groups.items():
        if code in codes:
            return group
    return None

# Appliquer la fonction sur la colonne 'Mother's occupation' et 'Father's occupation'
df["Mother's occupation Group"] = df["Mother's occupation"].apply(map_parents_occupation_group)
df["Father's occupation Group"] = df["Father's occupation"].apply(map_parents_occupation_group)

# Insérer la nouvelle colonne juste après 'Mother's qualification'
cols = list(df.columns)
idx = cols.index("Mother's occupation")
cols.insert(idx + 1, cols.pop(cols.index("Mother's occupation Group")))
df = df[cols]

# Insérer la nouvelle colonne juste après 'Father's qualification'
cols = list(df.columns)
idx = cols.index("Father's occupation")
cols.insert(idx + 1, cols.pop(cols.index("Father's occupation Group")))
df = df[cols]

# Afficher les informations sur les targets
print("\nDistribution des cibles dans le DataFrame brut :")
print(df['Target'].value_counts())
print("\nPourcentages:")
print((df['Target'].value_counts(normalize=True) * 100).round(1))

### Enregistrement du DataFrame modifié ###

# Définir le chemin du dossier et du fichier
file_path = data_folder / "data_update.csv"

# Sauvegarder le DataFrame modifié
df.to_csv(file_path, sep=';', index=False, encoding='utf-8')

# Séparation des données

In [7]:
# Charger les données
df = pd.read_csv(data_folder / 'data_update.csv', delimiter=';')

# =============================================================================
# CRÉER 2 FICHIERS SÉPARÉS
# =============================================================================

print("="*80)
print("SÉPARATION DU DATASET EN 2 FICHIERS")
print("="*80)

# FICHIER 1: Enrolled uniquement
df_enrolled = df[df['Target'] == 'Enrolled']
df_enrolled.to_csv(data_folder / 'data_enrolled.csv', index=False, sep=';')

print(f"\n✓ data_enrolled.csv")
print(f"  Nombre d'observations: {len(df_enrolled)}")
print(f"  Nombre de colonnes: {len(df_enrolled.columns)}")

# FICHIER 2: Graduate + Dropout
df_at_risk = df[df['Target'].isin(['Graduate', 'Dropout'])]
df_at_risk.to_csv(data_folder / 'data_graduate_dropout.csv', index=False, sep=';')

print(f"\n✓ data_graduate_dropout.csv")
print(f"  Nombre d'observations: {len(df_at_risk)}")
print(f"  Nombre de colonnes: {len(df_at_risk.columns)}")

# Vérifier
print(f"\n{'='*80}")
print("VÉRIFICATION")
print(f"{'='*80}")
print(f"\nOriginal dataset: {len(df)} observations")
print(f"Enrolled: {len(df_enrolled)} ({len(df_enrolled)/len(df)*100:.1f}%)")
print(f"Graduate + Dropout: {len(df_at_risk)} ({len(df_at_risk)/len(df)*100:.1f}%)")
print(f"Total: {len(df_enrolled) + len(df_at_risk)} ✓")

print(f"\nDistribution dans data_graduate_dropout.csv:")
print(df_at_risk['Target'].value_counts())
print("\nPourcentages:")
print((df_at_risk['Target'].value_counts(normalize=True) * 100).round(1))


SÉPARATION DU DATASET EN 2 FICHIERS

✓ data_enrolled.csv
  Nombre d'observations: 782
  Nombre de colonnes: 43

✓ data_graduate_dropout.csv
  Nombre d'observations: 3614
  Nombre de colonnes: 43

VÉRIFICATION

Original dataset: 4396 observations
Enrolled: 782 (17.8%)
Graduate + Dropout: 3614 (82.2%)
Total: 4396 ✓

Distribution dans data_graduate_dropout.csv:
Target
Graduate    2193
Dropout     1421
Name: count, dtype: int64

Pourcentages:
Target
Graduate    60.7
Dropout     39.3
Name: proportion, dtype: float64
